![BeliefLens](assets/belieflens-notebook-header.png)

# Apply a frozen BeliefLens measurement profile from LangChain

## Goal

Insert an already validated BeliefLens measurement into an existing LangChain or LangGraph workflow. A single `measurement_profile_id` identifies the frozen benchmark version, ontology, prompt family, model identity, JSON calibrator and acceptance certificate. LangChain invokes the profile; it does not refit it.

## What happens

1. Inspect the benchmark schema so the measured states are explicit.
2. Load the frozen measurement profile by identifier.
3. Submit one evidence record through a LangChain-compatible node.
4. Receive calibrated state probabilities, uncertainty checks, a certificate decision and a trace link.
5. Route the next graph action to continue, review or abstain.

Code inputs are hidden by default; click a cell's disclosure control to inspect them. Outputs remain visible.

**Further information:** [BeliefLens primer](https://belieflens.org/#/primer) · [Integration overview](https://belieflens.org/#/software) · [Platform documentation](https://demo.belieflens.org/docs)


## 1. Inspect the declared benchmark schema

The local manifest makes the scientific task visible. At runtime the server verifies the corresponding frozen profile and applies its bound calibrator. The calibrator is persisted as schema-versioned JSON coefficients inside a content-hashed validation artifact—not as an executable pickle.


In [ ]:
import json, os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path('examples/notebooks/finance')
manifest = json.loads((ROOT / 'data/offline_reproduction/inputs/SPY_SGOV_benchmark_manifest.json').read_text())
print('Benchmark schema:', manifest.get('schema_version'))
print('Declared states:', manifest.get('states') or manifest.get('ontology', {}).get('states'))


## 2. Configure one LangChain-compatible measurement node

Install the BeliefLens SDK with its optional LangChain dependency. The node calls `POST /v1/measurements`; BeliefLens owns the provider call so native token probabilities are not discarded by an intermediate abstraction.

```bash
pip install 'belieflens[langchain]'
```


In [ ]:
from belieflens import BeliefLens
from belieflens.integrations.langchain import BeliefLensMeasurementRunnable

client = BeliefLens(
    base_url=os.getenv('BELIEFLENS_BASE_URL', 'https://demo.belieflens.org'),
    api_key=os.environ['BELIEFLENS_API_KEY'],
)
measurement = BeliefLensMeasurementRunnable(
    client=client,
    measurement_profile_id=os.environ['BELIEFLENS_MEASUREMENT_PROFILE_ID'],
    provider_api_key=os.environ['OPENAI_API_KEY'],
)
profile = client.measurement_profile(os.environ['BELIEFLENS_MEASUREMENT_PROFILE_ID'])
print('Loaded profile:', profile['id'], 'hash:', profile['content_hash'])


## 3. Invoke it as a chain or LangGraph node

This is calibration **application**, not calibration fitting. One call returns calibrated state probabilities, uncertainty diagnostics, a certificate decision and a trace identifier. Set `RUN_PROVIDER_CALLS=1` only when you intend to make the provider calls required by the frozen prompt family.


In [ ]:
record = {
    'evidence_text': 'Equity breadth strengthened while implied volatility and credit spreads declined.',
    'observation_time': '2026-08-22T12:00:00Z',
}
if os.getenv('RUN_PROVIDER_CALLS') == '1':
    result = measurement.invoke(record)
    print(json.dumps({
        'state_probabilities': result['state_probabilities'],
        'decision': result['decision'],
        'measurement_id': result['measurement_id'],
        'observability': result['observability'],
    }, indent=2))
else:
    print('Dry run. Set RUN_PROVIDER_CALLS=1 to call the frozen measurement profile.')


## 4. Gate the next action

In LangGraph, use the returned decision before a trade, external tool or other consequential node. The gate does not claim that the proposed action is correct; it establishes whether the semantic measurement passed its declared conditions.


In [ ]:
def route_measurement(state):
    decision = state['measurement']['decision']
    if decision == 'measurement_accepted_within_scope':
        return 'continue'
    if 'abstain' in decision:
        return 'abstain'
    return 'human_review'

# graph.add_conditional_edges('measure', route_measurement, {
#     'continue': 'action', 'human_review': 'review', 'abstain': 'stop'
# })
